# V3.1 LIVE-SAFE — Grid LAG features only (tuned XGBoost, for the live pipeline)

The full V3.1 model (62 features) also used the 7 **current-period** grid flows
(`fi_ee` ... `fi_se_abs`). Those are NOT available at live forecast time for
future delivery slots, so they must be excluded from the production model.

This notebook trains the **live-safe** version: the 49 V2.5 features + the 6
grid **lag** features (`fi_total_net`/`fi_se_total`/`fi_ee` at `lag_96`=24h and
`lag_672`=7d) = **55 features**. Lags are computable from measured history at
inference, so this model can enter the daily `src/` pipeline.

Same setup: chronological 80/20, V2.5.3 tuned params (MAE loss, 2000 trees).

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Fix: Chinese Windows GBK -> sklearn HTML repr UnicodeDecodeError; force text display.
import sklearn
sklearn.set_config(display='text')

In [2]:
df = pd.read_csv('../data/convertData/V3_15min_features.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)
print('Shape:', df.shape)

Shape: (105216, 64)


In [3]:
# baseline = 49 original V2.5 features; enhanced = 49 + 6 grid LAG features (55)
baseline_cols = [c for c in df.columns
                 if c not in ['datetime', 'price'] and not c.startswith('fi_')]
lag_cols = [c for c in df.columns if c.startswith('fi_') and 'lag' in c]
enhanced_cols = baseline_cols + lag_cols

X_base = df[baseline_cols]   # 49 features
X_enh  = df[enhanced_cols]   # 55 features (49 + 6 grid lags)
y = df['price']
print(f'baseline: {len(baseline_cols)} features | enhanced (live-safe): {len(enhanced_cols)} features')
print('grid lag features:', lag_cols)

n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_base_train, X_base_test = X_base.iloc[:train_end], X_base.iloc[train_end:]
X_enh_train,  X_enh_test  = X_enh.iloc[:train_end],  X_enh.iloc[train_end:]
y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]
print(f'Train: {X_base_train.shape[0]}  Test: {X_base_test.shape[0]}')

baseline: 49 features | enhanced (live-safe): 55 features
grid lag features: ['fi_total_net_lag_96', 'fi_total_net_lag_672', 'fi_se_total_lag_96', 'fi_se_total_lag_672', 'fi_ee_lag_96', 'fi_ee_lag_672']
Train: 84173  Test: 21043


In [4]:
# V2.5.3 best hyperparameters (30-trial Optuna, MAE loss, 2000 trees)
tuned = dict(
    objective='reg:absoluteerror', n_estimators=2000,
    learning_rate=0.00982714905428372, max_depth=12, min_child_weight=31,
    subsample=0.7997314659075123, colsample_bytree=0.9982960915995492,
    reg_lambda=0.012943440208283537, reg_alpha=0.43805234879252597,
    random_state=42,
)

def train_eval(Xtr, ytr, Xte, yte):
    m = XGBRegressor(**tuned, verbosity=0)
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    return (mean_absolute_error(yte, p),
            np.sqrt(mean_squared_error(yte, p)),
            r2_score(yte, p))

base = train_eval(X_base_train, y_train, X_base_test, y_test)

# keep the live-safe model object (used by the save cell below)
model_v31_live = XGBRegressor(**tuned, verbosity=0)
model_v31_live.fit(X_enh_train, y_train)
p = model_v31_live.predict(X_enh_test)
enh = (mean_absolute_error(y_test, p),
       np.sqrt(mean_squared_error(y_test, p)),
       r2_score(y_test, p))

print('Both models trained (tuned XGBoost).')

Both models trained (tuned XGBoost).


In [5]:
comp = pd.DataFrame(
    {'XGBoost baseline (49)': base, 'XGBoost +grid-lags (55)': enh},
    index=['MAE', 'RMSE', 'R2']).T.round(4)
print(comp)

delta = enh[0] - base[0]
print('\nMAE delta (+grid-lags - baseline): %+.4f' % delta)
if delta < 0:
    print('Verdict: live-safe grid LAG features HELP under tuned XGBoost')
elif delta > 0:
    print('Verdict: grid LAG features still HURT even under tuned XGBoost')
else:
    print('Verdict: no difference')

print('\nReference (full V3.1, incl current-period flows):')
print('  baseline MAE 2.7236 -> +grid(62) MAE 2.7152 (delta -0.0084, helped)')

                            MAE    RMSE      R2
XGBoost baseline (49)    2.7236  8.1642  0.9722
XGBoost +grid-lags (55)  2.7416  8.2676  0.9715

MAE delta (+grid-lags - baseline): +0.0180
Verdict: grid LAG features still HURT even under tuned XGBoost

Reference (full V3.1, incl current-period flows):
  baseline MAE 2.7236 -> +grid(62) MAE 2.7152 (delta -0.0084, helped)


## Verdict — LIVE-SAFE grid features do NOT help

| Model | MAE | RMSE | R² |
| ----- | --- | ---- | --- |
| XGBoost baseline (49) | **2.7236** | 8.1642 | 0.9722 |
| XGBoost +grid-lags (55) | 2.7416 | 8.2676 | 0.9715 |

MAE delta: **+0.0180** (live-safe grid LAG features HURT slightly).

**Key finding — train/serve availability gap:**
- Full V3.1 (62 features, incl the 7 **current-period** flows): delta **−0.0084** (helped).
- Live-safe V3.1 (55 features, LAG-only): delta **+0.0180** (hurt).
- The grid signal that helps in training (current-period flows) is exactly the part
  that is NOT available at live forecast time; the live-feasible lags don't help.

**Decision: do NOT deploy a grid model.** The tuned XGBoost V2.5.3 remains the
production model. The `src/` grid infrastructure (GridBuffer + fetch_grid) is
kept, tested, and ready IF a live-feasible grid feature ever proves useful —
but the current V3.1 experiment cannot enter the daily pipeline.


In [6]:
# ── Save LIVE-SAFE model (only if it does not hurt) ──────────────────────────
import joblib
from pathlib import Path

if enh[0] <= base[0]:
    save_dir = Path('../models/saved')
    save_dir.mkdir(exist_ok=True)
    joblib.dump({
        'model': model_v31_live,
        'feature_cols': X_enh_train.columns.tolist(),
        'step_min': 15,
    }, save_dir / 'xgboost_v3_1.pkl')
    print('Saved → models/saved/xgboost_v3_1.pkl (live-safe, 55 features)')
else:
    print('Grid LAG features hurt the model -> NOT saved. Keeping tuned V2.5.3 only.')

Grid LAG features hurt the model -> NOT saved. Keeping tuned V2.5.3 only.
